# Cell type classification

In [2]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import warnings 
warnings.filterwarnings('ignore')

## Set input path

In [ ]:
base_path = '/home/jiahao/wanglab/Data/Analyzed/2024-12-02-Mingrui-SCZ'

input_path = os.path.join(base_path, "expr", 'integrated h5ad')
out_path = os.path.join(base_path, 'cell type classification')
if not os.path.exists(out_path):
    os.mkdir(out_path)
    
fig_path = os.path.join(out_path, 'figures')
if not os.path.exists(fig_path):
    os.mkdir(fig_path)

sc.settings.figdir = fig_path

In [ ]:
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-03-all-sample-combined-harmony-2-genotype-protocol-replicate.h5ad'))
adata

In [ ]:
sc.pl.umap(adata, color='replicate')
sc.pl.umap(adata, color='protocol')
sc.pl.umap(adata, color='genotype')

In [ ]:
unique_samples = adata.obs['genotype_protocol_replicate'].unique()

# Create a subplot grid
num_rows = 4
num_cols = 2
fig, axes = plt.subplots(num_rows, num_cols, figsize=(10,20))

# Flatten the axes array to iterate over each subplot
axes_flat = axes.flatten()

# Iterate over each unique sample
for i, sample in enumerate(unique_samples):
    # Subset the data for the current sample
    adata_sample = adata[adata.obs['genotype_protocol_replicate'] == sample].copy()

    # Plot UMAP for the current sample colored by sample
    sc.pl.umap(adata_sample, color='genotype_protocol_replicate', title=f'{sample}',legend_loc=None, show=False, ax=axes_flat[i])


In [ ]:
cluster_resolution = 2
sc.tl.leiden(adata, resolution=cluster_resolution)

In [10]:
sc.pl.umap(adata, color='leiden')

In [ ]:
sc.tl.rank_genes_groups(adata,'leiden', mask_var='highly_variable', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, dendrogram=False, standard_scale='var')

In [13]:
x_coords = adata.obs['global_x']
y_coords = adata.obs['global_y']

spatial_embedding = np.vstack((x_coords,y_coords)).T

adata.obsm['spatial'] = spatial_embedding

In [14]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata.write_h5ad(f"{out_path}/{date}-all-sample-leiden-res2.0.h5ad")

## Spatial pattern of Leiden clusters

In [ ]:
input_path = os.path.join(base_path, "expr", 'cell type classification')
cdata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster.h5ad'))
cdata

In [ ]:
input_path = os.path.join(base_path, "expr", 'cell type classification')
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-04-all-sample-leiden-res2.0.h5ad'))
adata

In [ ]:
cdata.obs['level_3'].value_counts()

In [ ]:
adata.obs['leiden'].value_counts()

In [ ]:
sc.pl.umap(cdata, color='level_3', title='leiden clusters', show=False)

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='leiden clusters', show=False)

In [15]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 10, num_rows*10))

# Flatten the axes array to iterate over each subplot
axes_flat = axes.flatten()

# Iterate over each unique sample
for i, rep in enumerate(['rep1','rep2']):
    for j, sample in enumerate(['WT_STAR', 'HET_STAR',  'WT_RIBO', 'HET_RIBO']):
        # Subset the data for the current sample
        adata_sample = adata[(adata.obs['replicate'] == rep) & (adata.obs['genotype_protocol'] == sample)].copy()

        # Plot UMAP for the current sample colored by sample
        sc.pl.spatial(adata_sample, color='leiden',spot_size = 150, title=f'{sample}_{rep}',legend_loc=None, show=False, ax=axes_flat[i*4+j])

In [26]:
clusters = adata.obs['leiden'].cat.categories.tolist()
color =  adata.uns['leiden_colors']

In [27]:
num_rows = 4
num_cols = len(clusters)+1
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

# Flatten the axes array to iterate over each subplot
axes_flat = axes.flatten()

for i, sample in enumerate(['WT_STAR', 'WT_RIBO', 'HET_STAR', 'HET_RIBO']):
#for i, sample in enumerate(['WT_STAR', 'WT_RIBO']):
    # Subset the data for the current sample
    adata_sample = adata[adata.obs['genotype_protocol'] == sample].copy()


    # Plot the UMAP plot with color by Leiden clusters
    sc.pl.spatial(adata_sample, color='leiden', spot_size=150, title = f'{sample}',legend_loc= None, show=False, ax=axes_flat[i*num_cols])
    

    # Plot for each cluster
    for c in range(0,len(clusters)):
        # Set other clusters to grey
        palette = {str(cluster): '#dddddd' for cluster in clusters if cluster != clusters[c]}
        palette[clusters[c]] = color[c]

        # Plot
        sc.pl.spatial(adata_sample, color='leiden', spot_size = 150, palette=palette, title = f'{clusters[c]}', legend_loc= None, show=False, ax=axes_flat[i*num_cols+c+1])

        # Show or save the plot
        #plt.show()
#plt.savefig(os.path.join(out_path, "spatial_pattern_of_leiden_clusters.png"), dpi=300)
plt.show()   

In [31]:
num_rows = 4
layer_markers = ['Cst3','Cux2','Calb1','Lamp5','Rorb','Pde1a']
num_cols = len(layer_markers)

fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))
axes_flat = axes.flatten()

for i, sample in enumerate(['WT_STAR', 'WT_RIBO', 'HET_STAR', 'HET_RIBO']):
#for i, sample in enumerate(['WT_STAR', 'WT_RIBO']):
    adata_sample = adata[adata.obs['genotype_protocol'] == sample].copy()
    for j, gene in enumerate(layer_markers):
        sc.pl.spatial(adata_sample, color=gene, spot_size = 150, legend_loc= None, show=False,ax=axes_flat[i*num_cols+j])

plt.show()

## Cell typing

In [58]:
# create backup for leiden label
adata.obs['orig_leiden'] = adata.obs['leiden'].values

adata.obs['level_1'] = adata.obs['leiden'].values
adata.obs['level_2'] = adata.obs['leiden'].values
adata.obs['level_3'] = adata.obs['leiden'].values

In [59]:
# Change cluster label to cell type label
transfer_dict_l1 = {}
transfer_dict_l2 = {}
transfer_dict_l3 = {}

In [60]:
# Level_1
level_1_list = [
    'Neuron', #0
    'Glia', #1
    'Glia', #2
    'Glia', #3
    'Neuron', #4
    'Neuron', #5
    'Neuron', #6
    'Neuron', #7
    'Neuron', #8
    'Glia', #9
    'Neuron', #10
    'Neuron', #11
    'Glia', #12
    'Neuron', #13
    'Neuron', #14
    'Glia', #15
    'Neuron', #16
    'Glia', #17
    'Glia', #18
    'Glia', #19
    'Glia', #20
    'Glia', #21
    'Neuron', #22
    'Glia', #23
    'Glia', #24
    'Neuron', #25
    'Neuron', #26
    'Glia', #27
]

for i in sorted(adata.obs['leiden'].unique()):
    transfer_dict_l1[i] = level_1_list[int(i)]

In [61]:
adata.obs = adata.obs.replace({'level_1': transfer_dict_l1})

In [62]:
# Level_2
level_2_list = [
    'TEGLU', #0
    'OLG', #1
    'AC', #2
    'PER', #3
    'TEGLU', #4
    'DE/MEGLU', #5
    'MSN', #6
    'MSN', #7
    'TEGLU', #8
    'MGL', #9
    'TEGLU', #10
    'TEINH', #11
    'OLG', #12
    'TEGLU', #13
    'TEGLU', #14
    'VEN', #15
    'TEINH', #16
    'OPC', #17
    'AC', #18
    'VSM', #19
    'CHOR', #20
    'VLM', #21
    'TEINH', #22
    'OLG', #23
    'AC', #24
    'PEP', #25
    'HABCHO', #26
    'VLM', #27
]
for i in sorted(adata.obs['leiden'].unique()):
    transfer_dict_l2[i] = level_2_list[int(i)]


In [63]:
adata.obs = adata.obs.replace({'level_2': transfer_dict_l2})

In [64]:

# Level_3
level_3_list = [
    'TEGLU-[Lamp5_Nrgn]', #0
    'OLG_1-[Opalin_Mal]', #1
    'AC_2-[Mfge8_Cspg5]/AC_3-[Mfge8_Cspg5_Gldc]', #2
    'PER-[Pltp_Flt1_Ly6a]', #3
    'TEGLU-[Slc17a7_Nrgn]', #4
    'DE/MEGLU-[Prkcd_Tcf7l2]', #5
    'MSN-[Drd1]', #6
    'MSN-[Drd2]', #7
    'TEGLU-[Hpcal4_Hs3st4_Pde1a]', #8
    'MGL_1-[Csf1r_Ctss_Tmem119]', #9
    'TEGLU-[Slc17a7_Nrgn]', #10
    'TEINH-[Pvalb_Gad1]', #11
    'OLG_2-[Klk6_Anln_Mal]', #12
    'TEGLU-[Gabra5_Nr3c2_Neurod6]', #13
    'TEGLU-[Tcerg1l_Tox]', #14
    'VEN_1-[Vtn_Rgs5_Abcc9]/VEN_2-[Vtn_Rgs5]', #15
    'TEINH-[Npy_Sst]', #16
    'OPC-[Pdgfra_Cspg5_C1ql1]', #17
    'AC_1-[Slc6a11_Agt_Itih3]', #18
    'VSM_1-[Myh11_Vim_Tagln]', #19
    'CHOR-[Ttr_Folr1]', #20
    'VLM_1-[Ptgds_Slc6a13_Igf2]/VLM_2-[Ptgds_Slc47a1_Mgp]', #21
    'TEINH-[Vip_Synpr]', #22
    'OLG_1-[Opalin_Mal]', #23
    'AC_5-[C4b_Gfap_Mbp]', #24
    'PEP-[Dlk1]', #25
    'HABCHO_2-[Gm5741_Nwd2_Gng8_Lrrc55_Tac2]/HABCHO_3-[Nwd2]', #26
    'VLM_1-[Ptgds_Slc6a13_Igf2]', #27
]


# construct transfer dict
for i in sorted(adata.obs['leiden'].unique()):
    transfer_dict_l3[i] = level_3_list[int(i)]

In [65]:
adata.obs = adata.obs.replace({'level_3': transfer_dict_l3})

In [ ]:
sc.pl.umap(adata,color='level_2',title=None)

In [ ]:
adata.obs['level_3'] = pd.Categorical(adata.obs['level_3'])
adata.obs['level_3'] = adata.obs['level_3'].cat.reorder_categories(sorted(adata.obs['level_3'].cat.categories), ordered=True)

# Now plot the UMAP
sc.pl.umap(adata, color='level_3')

In [ ]:
# # Convert categories to list to ensure they are ordered
# categories = adata.obs['level_2'].cat.categories.tolist() + adata_neuron.obs['level_2'].cat.categories.tolist() 

# # Set categories of the target column to match those of the source column
# adata.obs['level_2'] = adata.obs['level_2'].astype('category').cat.set_categories(categories)

# # Assign values to the target column
# for idx in adata_neuron.obs.index.tolist():
#     adata.obs.loc[int(idx), 'level_2'] = adata_neuron.obs.loc[idx, 'level_2']


In [70]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata.write_h5ad(f"{out_path}/{date}-all-sample-cell_typing-lv3.h5ad")

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata.write_h5ad(f"{out_path}/{date}-all-sample-cell-typing-lv3-subcluster.h5ad")

### TEGLU subtyping

In [ ]:
# input
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster.h5ad'))
adata.var['max_counts'] = adata.layers['raw'].toarray().max(axis=0)
adata

#### Step1

In [ ]:
# Subset
curr_cells = adata.obs['leiden'].isin(['0', '4', '8', '10', '13', '14'])
sdata = adata[curr_cells, :].copy()
sdata

In [ ]:
%%time
# redo pp 
sdata.X = sdata.layers['raw'].copy()
del sdata.layers['norm1e4']
del sdata.layers['log2_norm1e4']
del sdata.layers['log2_norm1e4_scaled']

sc.pp.normalize_total(sdata)
sc.pp.log1p(sdata)
sdata.raw = sdata
sc.pp.scale(sdata)
sdata.layers['scaled'] = sdata.X.copy()
sc.pp.regress_out(sdata, ['total_counts'])
sdata.layers['corrected'] = sdata.X.copy()

# Run PCA
# sdata.X = sdata.layers['corrected'].copy()
sc.tl.pca(sdata, svd_solver='full', use_highly_variable=True)

# Plot explained variance 
sc.pl.pca_variance_ratio(sdata, log=False)

# Plot PCA
sc.pl.pca(sdata, color='genotype_protocol_replicate')

import scanpy.external as sce
sce.pp.harmony_integrate(sdata, 'genotype_protocol_replicate')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
clustering_out_path = os.path.join(fig_path, f'{date}-level3-clustering')
if not os.path.exists(clustering_out_path):
    os.mkdir(clustering_out_path)

In [ ]:
sub_id = 'leiden'
sub_level_fig_path = os.path.join(clustering_out_path, sub_id)
if not os.path.exists(sub_level_fig_path):
    os.mkdir(sub_level_fig_path)

In [ ]:
# Embedding parameters
emb_dict = {
    'leiden': {'n_neighbors': 50, 'n_pcs': 10, 
              'min_dist': .1, 'spread': 3, 
              'cluster_resolution': 1, 'random_state': 0},
}

save_embedding = True

In [ ]:
# replace regular pca with integrated pca 
sdata.obsm['X_pca'] = sdata.obsm['X_pca_harmony'].copy()
sc.pl.pca_variance_ratio(sdata, log=False)
sc.pl.pca(sdata, color='genotype_protocol_replicate')

In [ ]:
%%time
# Computing the neighborhood graph
n_neighbors = emb_dict[sub_id]['n_neighbors']
n_pcs = emb_dict[sub_id]['n_pcs']
min_dist = emb_dict[sub_id]['min_dist']
spread = emb_dict[sub_id]['spread']

sc.pp.neighbors(sdata, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=0)

# Run UMAP
sc.tl.umap(sdata, min_dist=min_dist, spread=spread)

In [ ]:
%%time
# Run leiden cluster
cluster_resolution = emb_dict[sub_id]['cluster_resolution']
random_state = emb_dict[sub_id]['random_state']
sc.tl.leiden(sdata, resolution = cluster_resolution, random_state=random_state)

# Plot UMAP with cluster labels 
sc.pl.umap(sdata, color='leiden')
n_clusters = sdata.obs['leiden'].unique().shape[0]

if save_embedding:
    # Save log
    with open(f'{sub_level_fig_path}/log_{sub_id}.txt', 'w') as f:
        f.write(f"""Number of neighbor: {n_neighbors}
    Number of PC: {n_pcs}
    Resolution: {cluster_resolution}
    Random state: {random_state}
    Min-distance: {min_dist}
    Number of clusters: {n_clusters}""")

    # save embeddings
    np.savetxt(f'{sub_level_fig_path}/embedding_{sub_id}_umap.csv', sdata.obsm['X_umap'], delimiter=",")

In [ ]:
# Get colormap
cluster_pl = sns.color_palette("hls", n_clusters)
cluster_cmap = ListedColormap(cluster_pl.as_hex())
sns.palplot(cluster_pl)

In [ ]:
# Plot UMAP with cluster labels w/ new color
fig, ax = plt.subplots(figsize=(10,7))
sc.pl.umap(sdata, color='leiden', legend_loc='on data',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='Sub level clustering (leiden)', palette=cluster_pl, save=False, ax=ax)

In [ ]:
# move label to level 2 and check
adata.obs['level_3_temp'] = 'NA'
adata.obs.loc[adata.obs['leiden'].isin(['0', '4', '8', '10', '13', '14']), 'level_3_temp'] = sdata.obs.leiden.values
adata.obs['level_3_temp'] = adata.obs['level_3_temp'].astype('category')
temp_order = sdata.obs.leiden.cat.categories.to_list()
temp_order.append('NA')
adata.obs['level_3_temp'] = adata.obs['level_3_temp'].cat.reorder_categories(temp_order)
temp_pl = sns.color_palette(sdata.uns['leiden_colors'] + ['#ebebeb'])

In [ ]:
# Plot UMAP with all cell embedding
fig, ax = plt.subplots(figsize=(10,7))
sc.pl.umap(adata, color='level_3_temp', legend_loc='on data',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='Sub level clustering (leiden)', palette=temp_pl, save=False, ax=ax)

In [ ]:
# Add log layer
# sdata.layers['log_raw'] = np.log1p(sdata.layers['raw'])
# sc.pp.normalize_total(sdata, layer='log_raw')
# sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', layer='log_raw', pts=True, use_raw=False, n_genes=adata.shape[1])

# Find gene markers for each cluster
sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', pts=True, use_raw=True, n_genes=adata.shape[1])

# Filter markers
sc.tl.filter_rank_genes_groups(sdata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [ ]:
# Dot plot logfoldchanges
sc.pl.rank_genes_groups_dotplot(sdata, key='rank_genes_groups', n_genes=5, values_to_plot='logfoldchanges', min_logfoldchange=1, vmax=5, vmin=-5, cmap='bwr', dendrogram=False)

In [ ]:
# Dot plot mean expression (##)
sc.pl.rank_genes_groups_dotplot(sdata, key='rank_genes_groups_filtered', n_genes=5, dendrogram=False)

In [ ]:
# Print markers 
markers = []
temp = pd.DataFrame(sdata.uns['rank_genes_groups_filtered']['names']).head(15)
for i in range(temp.shape[1]):
    curr_col = temp.iloc[:, i].to_list()
    markers = markers + curr_col
    # print(i, curr_col)
    print(i)
    for j in curr_col:
        print(j, end=' ')
    print('')

In [ ]:
current_sample = 'sample11'

fig, ax = plt.subplots(figsize=(10, 12))
g = sns.scatterplot(x='global_x', y='global_y', color='#ebebeb', 
                    data=adata.obs.loc[adata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g = sns.scatterplot(x='global_x', y='global_y', hue='level_2', 
                    # palette=cluster_pl,
                    palette='Set1',
                    data=sdata.obs.loc[sdata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g.set_title(current_sample)
g.invert_yaxis()
g.axes.xaxis.set_visible(False)
g.axes.yaxis.set_visible(False)

In [ ]:
# plot summary plot for each cluster
sub_level_sum_path = os.path.join(sub_level_fig_path, f'r_{cluster_resolution}_summary_repp')
if not os.path.exists(sub_level_sum_path):
    os.mkdir(sub_level_sum_path)

for i, current_cluster in enumerate(tqdm(sorted(sdata.obs['leiden'].unique()))):
    
    # get dfs 
    df1 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample1', :]
    df2 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample5', :]
    df3 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample9', :]

    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(16, 16))
    axs = axs.flatten()

    # plot1
    g1 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample1', :], 
                        s=5,
                        ax=axs[0])

    g1.set_title('sample1')
    g1.invert_xaxis()
    g1.axes.xaxis.set_visible(False)
    g1.axes.yaxis.set_visible(False)


    h1 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df1.loc[df1['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[0])

    # plot2
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample5', :], 
                        s=5,
                        ax=axs[1])

    g2.set_title('sample5')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df2.loc[df2['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[1])

    # plot3
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample9', :], 
                        s=5,
                        ax=axs[2])

    g2.set_title('sample9')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df3.loc[df3['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[2])
                     
    size_factor = 200000
    # umap1
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[3], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample1')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap2
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[4], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample5')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap3
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[5], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample9')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))
    
    # umap3
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[6], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample1')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))

    # umap4
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[7], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample5')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
    # umap4
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[8], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample9')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
    plt.savefig(os.path.join(sub_level_sum_path, f'cluster_{current_cluster}.jpeg'))

#### Step2

In [ ]:
# Subset
curr_cells = sdata.obs['leiden'].isin(['3', '7', '8', '9', '10'])
bdata = sdata[curr_cells, :].copy()
bdata

In [ ]:
%%time
# redo pp 
bdata.X = bdata.layers['raw'].copy()

sc.pp.normalize_total(bdata)
sc.pp.log1p(bdata)
bdata.raw = bdata
sc.pp.scale(bdata)
bdata.layers['scaled'] = bdata.X.copy()
sc.pp.regress_out(bdata, ['total_counts'])
bdata.layers['corrected'] = bdata.X.copy()

# Run PCA
# sdata.X = sdata.layers['corrected'].copy()
sc.tl.pca(bdata, svd_solver='full', use_highly_variable=True)

# Plot explained variance 
sc.pl.pca_variance_ratio(bdata, log=False)

# Plot PCA
sc.pl.pca(bdata, color='genotype_protocol_replicate')

import scanpy.external as sce
sce.pp.harmony_integrate(bdata, 'genotype_protocol_replicate')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
clustering_out_path = os.path.join(fig_path, f'{date}-level3-clustering')
if not os.path.exists(clustering_out_path):
    os.mkdir(clustering_out_path)

In [ ]:
sub_id = 'leiden_step2'
sub_level_fig_path = os.path.join(clustering_out_path, sub_id)
if not os.path.exists(sub_level_fig_path):
    os.mkdir(sub_level_fig_path)

In [ ]:
# Embedding parameters
emb_dict = {
    'leiden_step2': {'n_neighbors': 50, 'n_pcs': 10, 
              'min_dist': .1, 'spread': 3, 
              'cluster_resolution': .8, 'random_state': 0},
}

save_embedding = True

In [ ]:
# replace regular pca with integrated pca 
bdata.obsm['X_pca'] = bdata.obsm['X_pca_harmony'].copy()
sc.pl.pca_variance_ratio(bdata, log=False)
sc.pl.pca(bdata, color='genotype_protocol_replicate')

In [ ]:
%%time
# Computing the neighborhood graph
n_neighbors = emb_dict[sub_id]['n_neighbors']
n_pcs = emb_dict[sub_id]['n_pcs']
min_dist = emb_dict[sub_id]['min_dist']
spread = emb_dict[sub_id]['spread']

sc.pp.neighbors(bdata, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=0)

# Run UMAP
sc.tl.umap(bdata, min_dist=min_dist, spread=spread)

In [ ]:
%%time
# Run leiden cluster
cluster_resolution = emb_dict[sub_id]['cluster_resolution']
random_state = emb_dict[sub_id]['random_state']
sc.tl.leiden(bdata, resolution = cluster_resolution, random_state=random_state)

# Plot UMAP with cluster labels 
sc.pl.umap(bdata, color='leiden')
n_clusters = bdata.obs['leiden'].unique().shape[0]

if save_embedding:
    # Save log
    with open(f'{sub_level_fig_path}/log_{sub_id}.txt', 'w') as f:
        f.write(f"""Number of neighbor: {n_neighbors}
    Number of PC: {n_pcs}
    Resolution: {cluster_resolution}
    Random state: {random_state}
    Min-distance: {min_dist}
    Number of clusters: {n_clusters}""")

    # save embeddings
    np.savetxt(f'{sub_level_fig_path}/embedding_{sub_id}_umap.csv', bdata.obsm['X_umap'], delimiter=",")

In [ ]:
# Get colormap
cluster_pl = sns.color_palette("hls", n_clusters)
cluster_cmap = ListedColormap(cluster_pl.as_hex())
sns.palplot(cluster_pl)

In [ ]:
# Plot UMAP with cluster labels w/ new color
fig, ax = plt.subplots(figsize=(10,7))
sc.pl.umap(bdata, color='leiden', legend_loc='on data',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='Sub level clustering (leiden)', palette=cluster_pl, save=False, ax=ax)

In [ ]:
# Add log layer
# sdata.layers['log_raw'] = np.log1p(sdata.layers['raw'])
# sc.pp.normalize_total(sdata, layer='log_raw')
# sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', layer='log_raw', pts=True, use_raw=False, n_genes=adata.shape[1])

# Find gene markers for each cluster
sc.tl.rank_genes_groups(bdata, 'leiden', method='wilcoxon', pts=True, use_raw=True, n_genes=adata.shape[1])

# Filter markers
sc.tl.filter_rank_genes_groups(bdata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [ ]:
# Dot plot logfoldchanges
sc.pl.rank_genes_groups_dotplot(bdata, key='rank_genes_groups', n_genes=5, values_to_plot='logfoldchanges', min_logfoldchange=1, vmax=5, vmin=-5, cmap='bwr', dendrogram=False)

In [ ]:
# Print markers 
markers = []
temp = pd.DataFrame(bdata.uns['rank_genes_groups_filtered']['names']).head(15)
for i in range(temp.shape[1]):
    curr_col = temp.iloc[:, i].to_list()
    markers = markers + curr_col
    # print(i, curr_col)
    print(i)
    for j in curr_col:
        print(j, end=' ')
    print('')

In [ ]:
current_sample = 'sample11'

fig, ax = plt.subplots(figsize=(10, 12))
g = sns.scatterplot(x='global_x', y='global_y', color='#ebebeb', 
                    data=adata.obs.loc[adata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g = sns.scatterplot(x='global_x', y='global_y', hue='level_2', 
                    # palette=cluster_pl,
                    palette='Set1',
                    data=sdata.obs.loc[sdata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g.set_title(current_sample)
g.invert_yaxis()
g.axes.xaxis.set_visible(False)
g.axes.yaxis.set_visible(False)

In [ ]:
# plot summary plot for each cluster
sub_level_sum_path = os.path.join(sub_level_fig_path, f'r_{cluster_resolution}_summary_repp')
if not os.path.exists(sub_level_sum_path):
    os.mkdir(sub_level_sum_path)

for i, current_cluster in enumerate(tqdm(sorted(bdata.obs['leiden'].unique()))):
    
    # get dfs 
    df1 = bdata.obs.loc[bdata.obs['sample_id'] == 'sample1', :]
    df2 = bdata.obs.loc[bdata.obs['sample_id'] == 'sample5', :]
    df3 = bdata.obs.loc[bdata.obs['sample_id'] == 'sample9', :]

    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
    axs = axs.flatten()

    # plot1
    g1 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample1', :], 
                        s=5,
                        ax=axs[0])

    g1.set_title('sample1')
    g1.invert_xaxis()
    g1.axes.xaxis.set_visible(False)
    g1.axes.yaxis.set_visible(False)


    h1 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df1.loc[df1['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[0])

    # plot2
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample5', :], 
                        s=5,
                        ax=axs[1])

    g2.set_title('sample5')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df2.loc[df2['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[1])

    # plot3
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample9', :], 
                        s=5,
                        ax=axs[2])

    g2.set_title('sample9')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df3.loc[df3['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[2])
                     
    size_factor = 200000
    # umap1
    ax = sc.pl.umap(bdata, show=False, color=None, alpha=1, size=(size_factor / bdata.n_obs), ax=axs[3], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(bdata[(bdata.obs["leiden"] == current_cluster) & (bdata.obs['sample_id'] == 'sample1')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / bdata.n_obs),
           title='', show=False, palette=sns.color_palette([bdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap2
    ax = sc.pl.umap(bdata, show=False, color=None, alpha=1, size=(size_factor / bdata.n_obs), ax=axs[4], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(bdata[(bdata.obs["leiden"] == current_cluster) & (bdata.obs['sample_id'] == 'sample5')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / bdata.n_obs),
           title='', show=False, palette=sns.color_palette([bdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap3
    ax = sc.pl.umap(bdata, show=False, color=None, alpha=1, size=(size_factor / bdata.n_obs), ax=axs[5], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(bdata[(bdata.obs["leiden"] == current_cluster) & (bdata.obs['sample_id'] == 'sample9')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / bdata.n_obs),
           title='', show=False, palette=sns.color_palette([bdata.uns['leiden_colors'][int(current_cluster)]]))
    
#     # umap3
#     ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[6], title='', palette=sns.color_palette(['#fafafa']))
#     sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample1')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
#            title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))

#     # umap4
#     ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[7], title='', palette=sns.color_palette(['#fafafa']))
#     sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample5')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
#            title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
#     # umap4
#     ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[8], title='', palette=sns.color_palette(['#fafafa']))
#     sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample9')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
#            title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
    plt.savefig(os.path.join(sub_level_sum_path, f'cluster_{current_cluster}.jpeg'))

In [ ]:
current_sample = 'sample11'

fig, ax = plt.subplots(figsize=(10, 12))
g = sns.scatterplot(x='global_x', y='global_y', color='#ebebeb', 
                    data=adata.obs.loc[adata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g = sns.scatterplot(x='global_x', y='global_y', hue='leiden', 
                    # palette=cluster_pl,
                    palette='Set1',
                    data=bdata.obs.loc[bdata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g.set_title(current_sample)
g.invert_yaxis()
g.axes.xaxis.set_visible(False)
g.axes.yaxis.set_visible(False)

In [ ]:
sdata.obs['leiden_temp'] = sdata.obs['leiden'].values
sdata.obs['leiden_temp'] = sdata.obs['leiden_temp'].astype(object)
sdata.obs.loc[bdata.obs.index[bdata.obs['leiden'].isin(['5', '6'])], 'leiden_temp'] = '5'
sdata.obs.loc[bdata.obs.index[bdata.obs['leiden'].isin(['0', '2', '3', '7'])], 'leiden_temp'] = 'Mix'
sdata.obs.loc[bdata.obs.index[bdata.obs['leiden'].isin(['1'])], 'leiden_temp'] = 'CA'
sdata.obs.loc[bdata.obs.index[bdata.obs['leiden'].isin(['4'])], 'leiden_temp'] = 'DG'

In [ ]:
sc.pl.umap(sdata, color='leiden_temp', legend_loc='on data',)

In [ ]:
# Add log layer
# sdata.layers['log_raw'] = np.log1p(sdata.layers['raw'])
# sc.pp.normalize_total(sdata, layer='log_raw')
# sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', layer='log_raw', pts=True, use_raw=False, n_genes=adata.shape[1])

# Find gene markers for each cluster
sc.tl.rank_genes_groups(sdata, 'level_3', method='wilcoxon', pts=True, use_raw=True, n_genes=adata.shape[1])

# Filter markers
sc.tl.filter_rank_genes_groups(sdata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [ ]:
# Dot plot logfoldchanges
sc.pl.rank_genes_groups_dotplot(sdata, key='rank_genes_groups', n_genes=10, values_to_plot='logfoldchanges', min_logfoldchange=1, vmax=5, vmin=-5, cmap='bwr', dendrogram=False)

In [ ]:
l1_transfer_dict = {

    'CA': 'Neuron',
    'DG': 'Neuron',
    '0': 'Neuron',
    '5': 'Neuron',
    '2': 'Neuron',
    '4': 'Neuron',
    '6': 'Neuron',
    'Mix': 'Neuron',
    '1': 'Mix'
}

l2_transfer_dict = {

    'CA': 'TEGLU',
    'DG': 'DGGRC',
    '0': 'TEGLU',
    '5': 'TEGLU',
    '2': 'TEGLU',
    '4': 'TEGLU',
    '6': 'TEGLU',
    'Mix': 'TEGLU',
    '1': 'Mix'
}

l3_transfer_dict = {

    'CA': 'TEGLU_CA-[Neurod6_Prkcg]',
    'DG': 'DGGRC-[Prox1_Nr3c2]',
    '0': 'TEGLU_L6-[Pcp4_Hs3st4]',
    '5': 'TEGLU_L5/6-[C1ql3_Tox]',
    '2': 'TEGLU_L4/5-[Rorb_Plcxd2]',
    '4': 'TEGLU_L2/3-[Lamp5_Cux2]',
    '6': 'TEGLU_L2/3-[Lamp5_Cux2]',
    'Mix': 'TEGLU_Mix',
    '1': 'Mix'
}


In [ ]:
sdata.obs['level_1'] = sdata.obs['leiden_temp'].map(l1_transfer_dict)
sdata.obs['level_2'] = sdata.obs['leiden_temp'].map(l2_transfer_dict)
sdata.obs['level_3'] = sdata.obs['leiden_temp'].map(l3_transfer_dict)

In [ ]:
sc.pl.umap(sdata, color='level_3', legend_loc='on data',)

In [ ]:
del sdata.uns['rank_genes_groups_filtered']
del sdata.uns['rank_genes_groups']

sdata.write_h5ad(os.path.join(input_path, '2025-04-07-TEGLU-subtyping.h5ad'))

### TEINH subtyping

In [ ]:
# input
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster.h5ad'))
adata.var['max_counts'] = adata.layers['raw'].toarray().max(axis=0)
adata

#### Step1

In [ ]:
# Subset
curr_cells = adata.obs['level_2'].isin(['TEINH'])
sdata = adata[curr_cells, :].copy()
sdata

In [ ]:
%%time
# redo pp 
sdata.X = sdata.layers['raw'].copy()
del sdata.layers['norm1e4']
del sdata.layers['log2_norm1e4']
del sdata.layers['log2_norm1e4_scaled']

sc.pp.normalize_total(sdata)
sc.pp.log1p(sdata)
sdata.raw = sdata
sc.pp.scale(sdata)
sdata.layers['scaled'] = sdata.X.copy()
sc.pp.regress_out(sdata, ['total_counts'])
sdata.layers['corrected'] = sdata.X.copy()

# Run PCA
# sdata.X = sdata.layers['corrected'].copy()
sc.tl.pca(sdata, svd_solver='full', use_highly_variable=True)

# Plot explained variance 
sc.pl.pca_variance_ratio(sdata, log=False)

# Plot PCA
sc.pl.pca(sdata, color='genotype_protocol_replicate')

import scanpy.external as sce
sce.pp.harmony_integrate(sdata, 'genotype_protocol_replicate')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
clustering_out_path = os.path.join(fig_path, f'{date}-level3-clustering')
if not os.path.exists(clustering_out_path):
    os.mkdir(clustering_out_path)

In [ ]:
sub_id = 'TEINH'
sub_level_fig_path = os.path.join(clustering_out_path, sub_id)
if not os.path.exists(sub_level_fig_path):
    os.mkdir(sub_level_fig_path)

In [ ]:
# Embedding parameters
emb_dict = {
    'TEINH': {'n_neighbors': 50, 'n_pcs': 10, 
              'min_dist': .1, 'spread': 3, 
              'cluster_resolution': .5, 'random_state': 0},
}

save_embedding = True

In [ ]:
# replace regular pca with integrated pca 
sdata.obsm['X_pca'] = sdata.obsm['X_pca_harmony'].copy()
sc.pl.pca_variance_ratio(sdata, log=False)
sc.pl.pca(sdata, color='genotype_protocol_replicate')

In [ ]:
%%time
# Computing the neighborhood graph
n_neighbors = emb_dict[sub_id]['n_neighbors']
n_pcs = emb_dict[sub_id]['n_pcs']
min_dist = emb_dict[sub_id]['min_dist']
spread = emb_dict[sub_id]['spread']

sc.pp.neighbors(sdata, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=0)

# Run UMAP
sc.tl.umap(sdata, min_dist=min_dist, spread=spread)

In [ ]:
%%time
# Run leiden cluster
cluster_resolution = emb_dict[sub_id]['cluster_resolution']
random_state = emb_dict[sub_id]['random_state']
sc.tl.leiden(sdata, resolution = cluster_resolution, random_state=random_state)

# Plot UMAP with cluster labels 
sc.pl.umap(sdata, color='leiden')
n_clusters = sdata.obs['leiden'].unique().shape[0]

if save_embedding:
    # Save log
    with open(f'{sub_level_fig_path}/log_{sub_id}.txt', 'w') as f:
        f.write(f"""Number of neighbor: {n_neighbors}
    Number of PC: {n_pcs}
    Resolution: {cluster_resolution}
    Random state: {random_state}
    Min-distance: {min_dist}
    Number of clusters: {n_clusters}""")

    # save embeddings
    np.savetxt(f'{sub_level_fig_path}/embedding_{sub_id}_umap.csv', sdata.obsm['X_umap'], delimiter=",")

In [ ]:
sc.pl.umap(sdata, color='Sst')

In [ ]:
# Get colormap
cluster_pl = sns.color_palette("hls", n_clusters)
cluster_cmap = ListedColormap(cluster_pl.as_hex())
sns.palplot(cluster_pl)

In [ ]:
# Plot UMAP with cluster labels w/ new color
fig, ax = plt.subplots(figsize=(10,7))
sc.pl.umap(sdata, color='leiden', legend_loc='on data',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='Sub level clustering (leiden)', palette=cluster_pl, save=False, ax=ax)

In [ ]:
# move label to level 2 and check
adata.obs['level_3_temp'] = 'NA'
adata.obs.loc[adata.obs['level_2'].isin(['TEINH']), 'level_3_temp'] = sdata.obs.leiden.values
adata.obs['level_3_temp'] = adata.obs['level_3_temp'].astype('category')
temp_order = sdata.obs.leiden.cat.categories.to_list()
temp_order.append('NA')
adata.obs['level_3_temp'] = adata.obs['level_3_temp'].cat.reorder_categories(temp_order)
temp_pl = sns.color_palette(sdata.uns['leiden_colors'] + ['#ebebeb'])

In [ ]:
# Plot UMAP with all cell embedding
fig, ax = plt.subplots(figsize=(10,7))
sc.pl.umap(adata, color='level_3_temp', legend_loc='on data',
           legend_fontsize=12, legend_fontoutline=2, frameon=False, 
           title='Sub level clustering (leiden)', palette=temp_pl, save=False, ax=ax)

In [ ]:
# Add log layer
# sdata.layers['log_raw'] = np.log1p(sdata.layers['raw'])
# sc.pp.normalize_total(sdata, layer='log_raw')
# sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', layer='log_raw', pts=True, use_raw=False, n_genes=adata.shape[1])

# Find gene markers for each cluster
sc.tl.rank_genes_groups(sdata, 'leiden', method='wilcoxon', pts=True, use_raw=True, n_genes=adata.shape[1])

# Filter markers
sc.tl.filter_rank_genes_groups(sdata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [ ]:
# Dot plot logfoldchanges
sc.pl.rank_genes_groups_dotplot(sdata, key='rank_genes_groups', n_genes=10, values_to_plot='logfoldchanges', min_logfoldchange=1, vmax=5, vmin=-5, cmap='bwr', dendrogram=False)

In [ ]:
# Dot plot mean expression (##)
sc.pl.rank_genes_groups_dotplot(sdata, key='rank_genes_groups_filtered', n_genes=5, dendrogram=False)

In [ ]:
# Print markers 
markers = []
temp = pd.DataFrame(sdata.uns['rank_genes_groups_filtered']['names']).head(15)
for i in range(temp.shape[1]):
    curr_col = temp.iloc[:, i].to_list()
    markers = markers + curr_col
    # print(i, curr_col)
    print(i)
    for j in curr_col:
        print(j, end=' ')
    print('')

In [ ]:
current_sample = 'sample16'

fig, ax = plt.subplots(figsize=(10, 12))
g = sns.scatterplot(x='global_x', y='global_y', color='#ebebeb', 
                    data=adata.obs.loc[adata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g = sns.scatterplot(x='global_x', y='global_y', hue='leiden', 
                    # palette=cluster_pl,
                    palette='Set1',
                    data=sdata.obs.loc[sdata.obs['sample_id'] == current_sample, :], 
                    s=5,
                    ax=ax)

g.set_title(current_sample)
g.invert_yaxis()
g.axes.xaxis.set_visible(False)
g.axes.yaxis.set_visible(False)

In [ ]:
# plot summary plot for each cluster
sub_level_sum_path = os.path.join(sub_level_fig_path, f'r_{cluster_resolution}_summary_repp')
if not os.path.exists(sub_level_sum_path):
    os.mkdir(sub_level_sum_path)

for i, current_cluster in enumerate(tqdm(sorted(sdata.obs['leiden'].unique()))):
    
    # get dfs 
    df1 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample1', :]
    df2 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample5', :]
    df3 = sdata.obs.loc[sdata.obs['sample_id'] == 'sample9', :]

    fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(16, 16))
    axs = axs.flatten()

    # plot1
    g1 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample1', :], 
                        s=5,
                        ax=axs[0])

    g1.set_title('sample1')
    g1.invert_xaxis()
    g1.axes.xaxis.set_visible(False)
    g1.axes.yaxis.set_visible(False)


    h1 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df1.loc[df1['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[0])

    # plot2
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample5', :], 
                        s=5,
                        ax=axs[1])

    g2.set_title('sample5')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df2.loc[df2['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[1])

    # plot3
    g2 = sns.scatterplot(x='global_x', y='global_y', color='#1111', 
                        data=adata.obs.loc[adata.obs['sample_id'] == 'sample9', :], 
                        s=5,
                        ax=axs[2])

    g2.set_title('sample9')
    g2.invert_yaxis()
    g2.axes.xaxis.set_visible(False)
    g2.axes.yaxis.set_visible(False)

    h2 = sns.scatterplot(x='global_x', y='global_y', hue='leiden', legend=None,
                        palette=cluster_pl,
                        data=df3.loc[df3['leiden'] == current_cluster, ], 
                        s=7,
                        ax=axs[2])
                     
    size_factor = 200000
    # umap1
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[3], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample1')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap2
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[4], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample5')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))

    # umap3
    ax = sc.pl.umap(sdata, show=False, color=None, alpha=1, size=(size_factor / sdata.n_obs), ax=axs[5], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(sdata[(sdata.obs["leiden"] == current_cluster) & (sdata.obs['sample_id'] == 'sample9')], color='leiden', frameon=False, ax=ax, legend_loc=None, size=(size_factor / sdata.n_obs),
           title='', show=False, palette=sns.color_palette([sdata.uns['leiden_colors'][int(current_cluster)]]))
    
    # umap3
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[6], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample1')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))

    # umap4
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[7], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample5')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
    # umap4
    ax = sc.pl.umap(adata, show=False, color=None, alpha=1, size=(size_factor / adata.n_obs), ax=axs[8], title='', palette=sns.color_palette(['#fafafa']))
    sc.pl.umap(adata[(adata.obs["level_3_temp"] == current_cluster) & (adata.obs['sample_id'] == 'sample9')], color='level_3_temp', frameon=False, ax=ax, legend_loc=None, size=(size_factor / adata.n_obs),
           title='', show=False, palette=sns.color_palette([adata.uns['level_3_temp_colors'][int(current_cluster)]]))
    
    plt.savefig(os.path.join(sub_level_sum_path, f'cluster_{current_cluster}.jpeg'))

In [ ]:
l1_transfer_dict = {

    '0': 'Neuron',
    '1': 'Neuron',
    '2': 'Mix',
    '3': 'Neuron',
    '4': 'Neuron',
    '5': 'Mix',
    '6': 'Mix'
}

l2_transfer_dict = {

    '0': 'TEINH',
    '1': 'TEINH',
    '2': 'Mix',
    '3': 'TEINH',
    '4': 'TEINH',
    '5': 'Mix',
    '6': 'Mix'

}

l3_transfer_dict = {

    '0': 'TEINH_1-[Pvalb_Gad1]',
    '1': 'TEINH_2-[Sst_Npy]',
    '2': 'Mix',
    '3': 'TEINH_3-[Lamp5_Npy]',
    '4': 'TEINH_4-[Vip_Cnr1]',
    '5': 'Mix',
    '6': 'Mix'

}


In [ ]:
sdata.obs['level_1'] = sdata.obs['leiden'].map(l1_transfer_dict)
sdata.obs['level_2'] = sdata.obs['leiden'].map(l2_transfer_dict)
sdata.obs['level_3'] = sdata.obs['leiden'].map(l3_transfer_dict)

In [ ]:
sc.pl.umap(sdata, color='level_3', legend_loc='on data',)

In [ ]:
del sdata.uns['rank_genes_groups_filtered']
del sdata.uns['rank_genes_groups']

sdata.write_h5ad(os.path.join(input_path, '2025-04-08-TEINH-subtyping.h5ad'))

## Finalize annotation

In [ ]:
# input
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster-aligned.h5ad'))
adata.var['max_counts'] = adata.layers['raw'].toarray().max(axis=0)
adata

In [ ]:
# input
gdata = sc.read_h5ad(os.path.join(input_path, '2025-04-07-TEGLU-subtyping.h5ad'))
gdata

In [ ]:
# input
idata = sc.read_h5ad(os.path.join(input_path, '2025-04-08-TEINH-subtyping.h5ad'))
idata

### Update level 1 & 2

In [ ]:
adata.obs['level_1'] = adata.obs['level_1'].astype(object)
adata.obs['level_2'] = adata.obs['level_2'].astype(object)

In [ ]:
adata.obs.loc[gdata.obs.index, 'level_1'] = gdata.obs['level_1'].values
adata.obs.loc[gdata.obs.index, 'level_2'] = gdata.obs['level_2'].values

adata.obs.loc[idata.obs.index, 'level_1'] = idata.obs['level_1'].values
adata.obs.loc[idata.obs.index, 'level_2'] = idata.obs['level_2'].values

In [ ]:
adata.obs['level_2'].unique()

### Update level 3

In [ ]:
adata.obs['level_3'] = adata.obs['leiden'].values

In [ ]:
adata.obs['level_3'] = adata.obs['level_3'].astype(object)

adata.obs.loc[gdata.obs.index, 'level_3'] = gdata.obs['level_3'].values
adata.obs.loc[idata.obs.index, 'level_3'] = idata.obs['level_3'].values

In [ ]:
adata.obs['level_3'].unique()

In [ ]:
# Find gene markers for each cluster
sc.tl.rank_genes_groups(adata, 'level_3', method='wilcoxon', pts=True, layer='log2_norm1e4', n_genes=adata.shape[1])

# Filter markers
sc.tl.filter_rank_genes_groups(adata, min_fold_change=.1, min_in_group_fraction=0.2, max_out_group_fraction=0.8)

In [ ]:
# Dot plot logfoldchanges
sc.pl.rank_genes_groups_dotplot(adata, key='rank_genes_groups', n_genes=10, values_to_plot='logfoldchanges', min_logfoldchange=1, vmax=5, vmin=-5, cmap='bwr', dendrogram=False)

In [ ]:
# Print markers 
markers = []
temp = pd.DataFrame(adata.uns['rank_genes_groups_filtered']['names']).head(15)
for i in range(temp.shape[1]):
    curr_col = temp.iloc[:, i].to_list()
    markers = markers + curr_col
    # print(i, curr_col)
    print(temp.columns[i])
    for j in curr_col:
        print(j, end=' ')
    print('')

In [ ]:
# Level_3
transfer_dict_l3 = {
'1': 'OLG_1-[Opalin_Mal]',
'2': 'AC_2-[Cspg5_Mfge8]',
'3': 'PER-[Pltp_Ly6a]',
'5': 'DEGLU-[Prkcd_Tcf7l2]',
'6': 'MSN_1-[Drd1_Rasd2]',
'7': 'MSN_2-[Drd2_Penk]',
'9': 'MGL-[Csf1r_P2ry12]',
'12': 'OLG_2-[Anln_Klk6]',
'15': 'VEN-[Vtn_Rgs5]',
'17': 'OPC-[Pdgfra_Cacng4]',
'18': 'AC_1-[Slc6a11_Agt]',
'19': 'VSM-[Myh11_Tagln]',
'20': 'CHOR-[Ttr_Folr1]',
'21': 'VLM_1-[Ptgds_Myoc]',
'23': 'OLG_3-[Mal_Cldn11]',
'24': 'AC_3-[Gfap_Aqp4]',
'25': 'PEP-[Dlk1_Ly6h]',
'26': 'HABCHO-[Lrrc55_Nwd2]',
'27': 'VLM_2-[Igf2_Igfbp2]',
}

In [ ]:
adata.obs['level_3'] = adata.obs['level_3'].replace(transfer_dict_l3)

In [ ]:
adata.obs['level_2'] = adata.obs['level_2'].replace({'DE/MEGLU': 'DEGLU'})

In [ ]:
l1_order = ['Neuron', 'Glia', 'Mix']
l2_order = ['TEGLU', 'DGGRC', 'TEINH', 'MSN', 'DEGLU', 'PEP', 'HABCHO',
            'AC', 'OLG', 'OPC', 'MGL', 
            'CHOR', 'PER', 'VEN', 'VLM', 'VSM', 'Mix']
l3_order = [
            'TEGLU_L2/3-[Lamp5_Cux2]', 'TEGLU_L4/5-[Rorb_Plcxd2]', 'TEGLU_L5/6-[C1ql3_Tox]',
            'TEGLU_L6-[Pcp4_Hs3st4]', 'TEGLU_CA-[Neurod6_Prkcg]', 'TEGLU_Mix', 'DGGRC-[Prox1_Nr3c2]',
            'TEINH_1-[Pvalb_Gad1]', 'TEINH_2-[Sst_Npy]', 'TEINH_3-[Lamp5_Npy]', 'TEINH_4-[Vip_Cnr1]',
            'MSN_1-[Drd1_Rasd2]', 'MSN_2-[Drd2_Penk]','DEGLU-[Prkcd_Tcf7l2]',
            'PEP-[Dlk1_Ly6h]', 'HABCHO-[Lrrc55_Nwd2]',
            'AC_1-[Slc6a11_Agt]', 'AC_2-[Cspg5_Mfge8]', 'AC_3-[Gfap_Aqp4]', 
            'OLG_1-[Opalin_Mal]','OLG_2-[Anln_Klk6]', 'OLG_3-[Mal_Cldn11]', 'OPC-[Pdgfra_Cacng4]',
            'MGL-[Csf1r_P2ry12]', 'CHOR-[Ttr_Folr1]', 'PER-[Pltp_Ly6a]',
            'VEN-[Vtn_Rgs5]', 'VLM_1-[Ptgds_Myoc]', 'VLM_2-[Igf2_Igfbp2]', 'VSM-[Myh11_Tagln]', 'Mix'
]

In [ ]:
adata.obs['level_1'] = adata.obs['level_1'].cat.reorder_categories(l1_order)
adata.obs['level_2'] = adata.obs['level_2'].cat.reorder_categories(l2_order)
adata.obs['level_3'] = adata.obs['level_3'].astype('category')
adata.obs['level_3'] = adata.obs['level_3'].cat.reorder_categories(l3_order)

In [ ]:
adata.uns = {}
adata.uns['level_1_order'] = l1_order
adata.uns['level_2_order'] = l2_order
adata.uns['level_3_order'] = l3_order

In [ ]:
adata.uns['level_1_color_dict'] = { 
    'Neuron': '#eded58',
    'Glia': '#356be8',
    'Mix': '#dddddd'
}

adata.uns['level_2_color_dict'] = {
    'TEGLU': '#3cb03c',
    'DGGRC': '#295029',
    'TEINH': '#ff7f0e',
    'MSN': '#17becf',
    'DEGLU': '#f78a88',
    'PEP': '#ed5e5b',
    'HABCHO': '#f29ed8',
    'AC': '#dbdb8d',
    'OLG': '#9edae5',
    'OPC': '#667872',
    'MGL': '#a2b1d8',
    'CHOR': '#7f52a9',
    'PER': '#c49c94',
    'VEN': '#8e6d61',
    'VLM': '#1f76b3',
    'VSM': '#774d44',
    'Mix': '#dddddd'
}

adata.uns['level_3_color_dict'] = {
    'TEGLU_L2/3-[Lamp5_Cux2]': '#c4ff45',
    'TEGLU_L4/5-[Rorb_Plcxd2]': '#9ee800',
    'TEGLU_L5/6-[C1ql3_Tox]': '#32a630',
    'TEGLU_L6-[Pcp4_Hs3st4]': '#316e10',
    'TEGLU_CA-[Neurod6_Prkcg]': '#00e846',
    'TEGLU_Mix': '#2c642c',
    'DGGRC-[Prox1_Nr3c2]': '#295029',
    'TEINH_1-[Pvalb_Gad1]': '#fcd02d',
    'TEINH_2-[Sst_Npy]': '#fead65',
    'TEINH_3-[Lamp5_Npy]': '#ffdcbd',
    'TEINH_4-[Vip_Cnr1]': '#b76319',
    'MSN_1-[Drd1_Rasd2]': '#7aecf8',
    'MSN_2-[Drd2_Penk]':'#06a6cf',
    'DEGLU-[Prkcd_Tcf7l2]':'#f78a88',
    'PEP-[Dlk1_Ly6h]':'#ed5e5b',
    'HABCHO-[Lrrc55_Nwd2]':'#f29ed8',
    'AC_1-[Slc6a11_Agt]':'#eaeaa2',
    'AC_2-[Cspg5_Mfge8]':'#bcbc5e',
    'AC_3-[Gfap_Aqp4]':'#a6a64d', 
    'OLG_1-[Opalin_Mal]':'#a8e1eb',
    'OLG_2-[Anln_Klk6]':'#7dc7d5', 
    'OLG_3-[Mal_Cldn11]':'#61b2c1',
    'OPC-[Pdgfra_Cacng4]':'#667872',
    'MGL-[Csf1r_P2ry12]':'#a2b1d8',
    'CHOR-[Ttr_Folr1]':'#7f52a9',
    'PER-[Pltp_Ly6a]':'#c49c94',
    'VEN-[Vtn_Rgs5]':'#8e6d61',
    'VLM_1-[Ptgds_Myoc]':'#1f76b3',
    'VLM_2-[Igf2_Igfbp2]':'#246693',
    'VSM-[Myh11_Tagln]':'#774d44',
    'Mix':'#dddddd'
}

In [ ]:
l1_colors = [adata.uns['level_1_color_dict'][i] for i in adata.obs['level_1'].cat.categories]
l2_colors = [adata.uns['level_2_color_dict'][i] for i in adata.obs['level_2'].cat.categories]
l3_colors = [adata.uns['level_3_color_dict'][i] for i in adata.obs['level_3'].cat.categories]

level_1_pl = sns.color_palette(l1_colors)
level_2_pl = sns.color_palette(l2_colors)
level_3_pl = sns.color_palette(l3_colors)

In [ ]:
adata_pfc = adata[adata.obs['coronal_position'] == 'ST'].copy()

num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_pfc.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_pfc[adata_pfc.obs['sample_id']==sample], color='level_3', spot_size=200, legend_loc=None, show=False, title=f'{sample}', ax=axes_flat[i])

In [ ]:
sc.pl.umap(adata, color='level_3', palette=level_3_pl)

In [ ]:
sc.pl.umap(adata, color='level_2', palette=level_2_pl)

In [ ]:
sc.pl.umap(adata, color='level_1', palette=level_1_pl)

In [ ]:
adata

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata.write_h5ad(f"{input_path}/{date}-finalized-celltyping.h5ad")